## Step 7 --- Evaluation

In [ ]:
#Importing all the necessary libraries
import os
import sys
import joblib
import shutil
import importlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
#Converting the processed training data into a dataframe
training_data = pd.read_csv(r"data/processed/training_data.csv")
training_data.head()
training_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   open_time           800 non-null    object 
 1   open                800 non-null    float64
 2   high                800 non-null    float64
 3   low                 800 non-null    float64
 4   close               800 non-null    float64
 5   volume              800 non-null    float64
 6   close_time          800 non-null    object 
 7   quote_asset_volume  800 non-null    float64
 8   num_trades          800 non-null    float64
 9   taker_base_volume   800 non-null    float64
 10  taker_quote_volume  800 non-null    float64
 11  return_1d           800 non-null    float64
 12  return_7d           800 non-null    float64
 13  rolling_volatility  800 non-null    float64
 14  rsi                 800 non-null    float64
 15  sma_20              800 non-null    float64
 16  sma_50  

In [ ]:
#Adding 'src' to the path
sys.path.append(os.path.abspath('../src'))

Import the evaluation script
import evaluate

Force reload
importlib.reload(evaluate)

2. EXECUTE EVALUATION ARENA
print(" STARTING EVALUATION ARENA...")
print("   - Loading models from 'models/'...")
print("   - Testing on the LAST 15% of data (Strictly Unseen).")
print("   - Comparing Accuracy and picking a winner.\n")

Run the function
evaluate.evaluate_models()

In [ ]:
#Creating a function to handle model evaluation
def evaluate_models():
    current_script_dir = os.path.dirname(os.path.abspath(_file_))
    project_root = os.path.dirname(current_script_dir)

    data_path = os.path.join(project_root, 'data', 'processed', 'training_data.csv')
    model_dir = os.path.join(project_root, 'models')

    print(" Starting Model Evaluation Arena...")

    #Preparing the test data
    if not os.path.exists(data_path):
        print(f" Error: Data not found at {data_path}")
        # FIX: Return 3 Nones so the notebook doesn't crash
        return None, None, None

    df = pd.read_csv(data_path)
    df.dropna(inplace=True)

    drop_cols = ['open_time', 'close_time', 'ignore', 'future_return', 'label', 'threshold_buy', 'threshold_sell']
    feature_cols = [c for c in df.columns if c not in drop_cols]

    X = df[feature_cols]
    y = df['label']

    test_start = int(len(df) * 0.85)
    X_test = X.iloc[test_start:]
    y_test = y.iloc[test_start:]

    print(f"   Testing on {len(X_test)} rows (Last 15%)\n")

    #Creating and initializing variables to track the best model
    best_acc = -1
    best_model_name = None
    best_preds = None

    # Checking for the directory
    if not os.path.exists(model_dir):
        print(f" Error: Models folder not found at {model_dir}")
        return None, None, None

    model_files = [f for f in os.listdir(model_dir) if f.endswith('.pkl') and f != 'best_crypto_model.pkl']

    if not model_files:
        print(" No models found! Run train.py first.")
        # FIX: Return 3 Nones
        return None, None, None

    print(f"{'MODEL':<20} | {'ACCURACY':<10}")
    print("-" * 35)

    for filename in model_files:
        model_name = filename.replace('.pkl', '')
        model_path = os.path.join(model_dir, filename)

        try:
            model = joblib.load(model_path)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)

            print(f"{model_name:<20} | {acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_model_name = model_name
                best_preds = preds

        except Exception as e:
            print(f" Error loading {model_name}: {e}")

    # --- 4. SHOW WINNER ---
    if best_model_name is None:
        print(" No valid models could be evaluated.")
        return None, None, None

    print("-" * 35)
    print(f" WINNER: {best_model_name} (Accuracy: {best_acc:.4f})")

    # --- 5. SAVE BEST MODEL ---
    src_file = os.path.join(model_dir, f"{best_model_name}.pkl")
    dst_file = os.path.join(model_dir, "best_crypto_model.pkl")
    try:
        shutil.copyfile(src_file, dst_file)
        print(f" Copied winner to: {dst_file}")
    except Exception as e:
        print(f" Could not copy best model: {e}")


 # --- 6. TEXT REPORT ---
    print("\n Classification Report (Winner):")
    print(classification_report(y_test, best_preds, target_names=['SELL', 'HOLD', 'BUY']))

    return best_model_name, y_test, best_preds

if name == "main":
    winner_name, y_true, y_pred = evaluate_models()

    if winner_name is not None:
        print("\n Generating Confusion Matrix Plot...")
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(7, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                    xticklabels=['Pred SELL', 'Pred HOLD', 'Pred BUY'],
                    yticklabels=['Act SELL', 'Act HOLD', 'Act BUY'])
        plt.title(f'Confusion Matrix: {winner_name}')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.show()

## Step 8 --- Serialize the Model

In [ ]:
import joblib
from keras.models import model
joblib.dump(model, "models/buy_sell_classifier.pkl")

## Step 9 --- Prediction Pipeline

In [ ]:
import joblib

def predict(features):
    model = joblib.load("models/buy_sell_classifier.pkl")
    return model.predict(features)